In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from IPython.display import Video, display
import time
from datetime import timedelta

# ==========================================
# CONFIGURATION & KAGGLE PATHS
# ==========================================
VIDEO_INPUT = "/kaggle/input/datasets/tanzinabdul/kl-road-video/sample.mov"
OUTPUT_DIR = "/kaggle/working/transformer_censored_video"
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "censored_video_1080p.mp4")
TEMP_FRAMES_DIR = os.path.join(OUTPUT_DIR, "temp_frames")

# Prompt covers vehicles, license plates, and faces/heads
TEXT_PROMPT = "vehicle . car . motorcycle . bus . truck . license plate . number plate . human face . head ."

# Detection Thresholds
BOX_THRESHOLD = 0.20      # Sensitive enough to catch small/distant targets
TEXT_THRESHOLD = 0.20

# Expansion Padding
PAD_PLATE_X = 0.10        # 10% horizontal expansion for wide/angled plates
PAD_PLATE_Y = 0.5         # 5% vertical expansion
PAD_FACE = 0.10           # 10% expansion around faces/heads

# Video processing settings
TARGET_WIDTH = 1920  # 1080p width
TARGET_HEIGHT = 1080 # 1080p height
FRAME_SKIP = 1       # Process every frame (1 = all frames, 2 = every other, etc.)
OUTPUT_FPS = 30      # Output video FPS

# ==========================================
# HARDWARE ACCELERATION (Multi-GPU Setup)
# ==========================================
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"[INFO] Found {num_gpus} GPU(s)")
    for i in range(num_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    
    device = "cuda"
    print(f"[INFO] Using all {num_gpus} GPUs with DataParallel")
else:
    device = "cpu"
    num_gpus = 0
    print(f"[INFO] No GPUs found, using CPU")

# ==========================================
# LOAD TRANSFORMER MODEL
# ==========================================
MODEL_ID = "IDEA-Research/grounding-dino-base"
print(f"[INFO] Loading Grounding DINO Transformer ({MODEL_ID})...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID)

# Move model to GPU(s)
if num_gpus > 0:
    model = model.to(device)
    if num_gpus > 1:
        model = torch.nn.DataParallel(model)
        print(f"[INFO] Model wrapped with DataParallel on {num_gpus} GPUs")

# ==========================================
# HELPER FUNCTIONS (from original notebook)
# ==========================================
def is_point_inside_box(point, box, margin_pct=0.10):
    """Checks if a point (cx, cy) is inside a vehicle box with a safety margin."""
    px, py = point
    vx1, vy1, vx2, vy2 = box
    vw = vx2 - vx1
    vh = vy2 - vy1
    
    # Add a small margin around vehicle boundary
    vx1 -= vw * margin_pct
    vy1 -= vy1 * margin_pct
    vx2 += vw * margin_pct
    vy2 += vh * margin_pct
    
    return (vx1 <= px <= vx2) and (vy1 <= py <= vy2)

def apply_blur(cv_image, box, pad_x_pct=0.20, pad_y_pct=0.15):
    """Applies Gaussian blur to bounding box with padding."""
    h, w, _ = cv_image.shape
    x1, y1, x2, y2 = map(int, box)
    
    box_w = x2 - x1
    box_h = y2 - y1
    
    pad_x = int(box_w * pad_x_pct)
    pad_y = int(box_h * pad_y_pct)
    
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)
    
    if x2 <= x1 or y2 <= y1:
        return cv_image
        
    roi = cv_image[y1:y2, x1:x2]
    
    kw = max(3, (roi.shape[1] // 2) | 1)
    kh = max(3, (roi.shape[0] // 2) | 1)
    
    blurred_roi = cv2.GaussianBlur(roi, (kw, kh), 0)
    cv_image[y1:y2, x1:x2] = blurred_roi
    return cv_image

def add_detection_overlay(frame, boxes, labels, scores, vehicle_boxes):
    """Add visual overlays to show what's being detected and censored."""
    overlay = frame.copy()
    
    # Color coding
    VEHICLE_COLOR = (0, 255, 0)    # Green
    PLATE_COLOR = (0, 0, 255)      # Red
    FACE_COLOR = (255, 255, 0)     # Cyan
    CENSOR_COLOR = (0, 255, 255)   # Yellow for censored items
    
    VEHICLE_KEYWORDS = ["vehicle", "car", "motorcycle", "bus", "truck"]
    PLATE_KEYWORDS = ["license plate", "number plate", "plate"]
    FACE_KEYWORDS = ["human face", "face", "head"]
    
    # Draw bounding boxes with labels
    for box, label, score in zip(boxes, labels, scores):
        lbl = label.lower()
        x1, y1, x2, y2 = map(int, box)
        
        # Determine color based on type
        if any(v in lbl for v in VEHICLE_KEYWORDS):
            color = VEHICLE_COLOR
            label_text = "Vehicle"
        elif any(p in lbl for p in PLATE_KEYWORDS):
            color = PLATE_COLOR
            label_text = "Plate"
        elif any(f in lbl for f in FACE_KEYWORDS):
            color = FACE_COLOR
            label_text = "Face"
        else:
            color = (128, 128, 128)  # Gray
            label_text = "Other"
        
        # Draw box
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        
        # Add label with confidence
        label_text = f"{label_text} {score:.2f}"
        label_size = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
        cv2.rectangle(overlay, (x1, y1 - label_size[1] - 10), (x1 + label_size[0], y1), color, -1)
        cv2.putText(overlay, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    # Add status text
    stats_text = [
        f"Vehicles: {len(vehicle_boxes)}",
        f"Plates detected: {sum(1 for l in labels if any(p in l.lower() for p in PLATE_KEYWORDS))}",
        f"Faces detected: {sum(1 for l in labels if any(f in l.lower() for f in FACE_KEYWORDS))}"
    ]
    
    y_offset = 30
    for i, text in enumerate(stats_text):
        cv2.putText(overlay, text, (10, y_offset + i*25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return overlay

def draw_censored_indicator(frame, applied_blurs):
    """Add a visual indicator for censored areas."""
    if applied_blurs > 0:
        cv2.putText(frame, f"CENSORED: {applied_blurs} regions", (10, 130), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    # Add semi-transparent "PRIVACY PROTECTED" overlay
    overlay = frame.copy()
    cv2.rectangle(overlay, (frame.shape[1] - 200, 10), (frame.shape[1] - 10, 40), (0, 0, 0), -1)
    cv2.putText(overlay, "PRIVACY", (frame.shape[1] - 190, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(overlay, "PROTECTED", (frame.shape[1] - 180, 50), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    
    # Blend overlay with transparency
    alpha = 0.3
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)
    return frame

# ==========================================
# VIDEO PROCESSING
# ==========================================
def process_video():
    """Main video processing function with progress tracking."""
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(TEMP_FRAMES_DIR, exist_ok=True)
    
    # Check if input video exists
    if not os.path.exists(VIDEO_INPUT):
        print(f"[ERROR] Video not found: {VIDEO_INPUT}")
        return
    
    # Open video capture
    cap = cv2.VideoCapture(VIDEO_INPUT)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {VIDEO_INPUT}")
        return
    
    # Get video properties
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"\n[INFO] Video Info:")
    print(f"  - Resolution: {original_width}x{original_height}")
    print(f"  - FPS: {original_fps:.2f}")
    print(f"  - Total frames: {total_frames}")
    print(f"  - Duration: {total_frames/original_fps:.2f} seconds")
    print(f"  - Output resolution: {TARGET_WIDTH}x{TARGET_HEIGHT}")
    
    # Calculate frames to process (with skip)
    frames_to_process = max(1, total_frames // FRAME_SKIP)
    print(f"  - Frames to process: {frames_to_process}")
    print("=" * 80)
    
    # Category keywords
    VEHICLE_KEYWORDS = ["vehicle", "car", "motorcycle", "bus", "truck"]
    PLATE_KEYWORDS = ["license plate", "number plate", "plate"]
    FACE_KEYWORDS = ["human face", "face", "head"]
    
    # Create video writer for 1080p output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, OUTPUT_FPS, (TARGET_WIDTH, TARGET_HEIGHT))
    
    if not out_writer.isOpened():
        print("[ERROR] Could not create video writer")
        cap.release()
        return
    
    # Processing statistics
    processed_frames = 0
    total_blurs = 0
    total_vehicles = 0
    start_time = time.time()
    
    # Progress bar for frame processing
    print("\n[INFO] Starting video processing...")
    pbar = tqdm(total=frames_to_process, desc="Processing frames", unit="frame")
    
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Skip frames based on FRAME_SKIP
        if frame_count % FRAME_SKIP != 0:
            frame_count += 1
            continue
        
        # Resize to 1080p
        frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT), interpolation=cv2.INTER_LANCZOS4)
        
        # Convert BGR to RGB for PIL/Transformer
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(frame_rgb)
        
        # Process with Transformer
        inputs = processor(images=pil_img, text=TEXT_PROMPT, return_tensors="pt")
        
        if num_gpus > 0:
            inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            if num_gpus > 1:
                outputs = model(**inputs)
            else:
                outputs = model(**inputs)
        
        target_sizes = [pil_img.size[::-1]]
        
        try:
            results = processor.post_process_grounded_object_detection(
                outputs, inputs["input_ids"], box_threshold=BOX_THRESHOLD, 
                text_threshold=TEXT_THRESHOLD, target_sizes=target_sizes
            )[0]
        except TypeError:
            results = processor.post_process_grounded_object_detection(
                outputs, inputs["input_ids"], threshold=BOX_THRESHOLD, 
                text_threshold=TEXT_THRESHOLD, target_sizes=target_sizes
            )[0]
        
        boxes = results["boxes"].cpu().numpy()
        scores = results["scores"].cpu().numpy()
        labels = results["labels"]
        
        # Classify detections
        vehicle_boxes = []
        plate_candidates = []
        face_candidates = []
        
        for box, score, label in zip(boxes, scores, labels):
            lbl = label.lower()
            if any(v in lbl for v in VEHICLE_KEYWORDS):
                vehicle_boxes.append(box)
            elif any(p in lbl for p in PLATE_KEYWORDS):
                plate_candidates.append((box, score, label))
            elif any(f in lbl for f in FACE_KEYWORDS):
                face_candidates.append((box, score, label))
        
        applied_blurs = 0
        
        # Filter and blur plates on vehicles
        for box, score, label in plate_candidates:
            px1, py1, px2, py2 = box
            plate_center = ((px1 + px2) / 2.0, (py1 + py2) / 2.0)
            
            is_on_vehicle = any(is_point_inside_box(plate_center, v_box) for v_box in vehicle_boxes)
            
            if is_on_vehicle or (len(vehicle_boxes) == 0 and score > 0.35):
                frame = apply_blur(frame, box, PAD_PLATE_X, PAD_PLATE_Y)
                applied_blurs += 1
        
        # Blur faces and heads
        for box, score, label in face_candidates:
            frame = apply_blur(frame, box, PAD_FACE, PAD_FACE)
            applied_blurs += 1
        
        # Add visual enhancements
        frame = add_detection_overlay(frame, boxes, labels, scores, vehicle_boxes)
        frame = draw_censored_indicator(frame, applied_blurs)
        
        # Add progress info on frame
        elapsed = time.time() - start_time
        progress_pct = (processed_frames / frames_to_process) * 100
        time_remaining = (elapsed / max(1, processed_frames)) * (frames_to_process - processed_frames)
        
        progress_text = f"Frame: {processed_frames}/{frames_to_process} ({progress_pct:.1f}%)"
        time_text = f"Time remaining: {str(timedelta(seconds=int(time_remaining)))}"
        
        cv2.putText(frame, progress_text, (10, TARGET_HEIGHT - 50), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        cv2.putText(frame, time_text, (10, TARGET_HEIGHT - 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # Write frame to output video
        out_writer.write(frame)
        
        # Update statistics
        processed_frames += 1
        total_blurs += applied_blurs
        total_vehicles += len(vehicle_boxes)
        
        # Update progress bar
        pbar.update(1)
        pbar.set_postfix_str(
            f"Blurs: {applied_blurs} | Vehicles: {len(vehicle_boxes)} | "
            f"ETA: {str(timedelta(seconds=int(time_remaining)))}"
        )
        
        frame_count += 1
    
    # Cleanup
    cap.release()
    out_writer.release()
    pbar.close()
    
    # Print summary
    total_time = time.time() - start_time
    print("\n" + "=" * 80)
    print(f"✅ Video processing complete!")
    print(f"  - Processed frames: {processed_frames}")
    print(f"  - Total blurs applied: {total_blurs}")
    print(f"  - Average blurs per frame: {total_blurs/max(1, processed_frames):.2f}")
    print(f"  - Total vehicles detected: {total_vehicles}")
    print(f"  - Processing time: {str(timedelta(seconds=int(total_time)))}")
    print(f"  - Output video: {OUTPUT_VIDEO}")
    print(f"  - Output resolution: {TARGET_WIDTH}x{TARGET_HEIGHT}")
    print("=" * 80)
    
    return OUTPUT_VIDEO

# ==========================================
# EXECUTION & PREVIEW
# ==========================================
if __name__ == "__main__":
    # Process the video
    output_video_path = process_video()
    
    # Display the video in notebook
    if output_video_path and os.path.exists(output_video_path):
        print(f"\n🎬 Video saved at: {output_video_path}")
        
        # Create a downloadable link
        from IPython.display import FileLink, display
        print("\n📥 Download the processed video:")
        display(FileLink(output_video_path))
        
        # Preview the video (first 5 seconds)
        try:
            from IPython.display import Video as IPVideo
            print("\n🎥 Preview of processed video (first 30 seconds):")
            # Show first 30 seconds or whole video if shorter
            preview_duration = min(30, int(cv2.VideoCapture(output_video_path).get(cv2.CAP_PROP_FRAME_COUNT) / OUTPUT_FPS))
            display(IPVideo(output_video_path, width=640, embed=True))
        except:
            print("Video preview not available in this environment")

[INFO] Found 2 GPU(s)
  GPU 0: Tesla T4
  GPU 1: Tesla T4
[INFO] Using all 2 GPUs with DataParallel
[INFO] Loading Grounding DINO Transformer (IDEA-Research/grounding-dino-base)...


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

[INFO] Model wrapped with DataParallel on 2 GPUs

[INFO] Video Info:
  - Resolution: 1920x1080
  - FPS: 29.64
  - Total frames: 1343
  - Duration: 45.30 seconds
  - Output resolution: 1920x1080
  - Frames to process: 1343

[INFO] Starting video processing...


Processing frames:   0%|          | 0/1343 [00:00<?, ?frame/s]/usr/local/lib/python3.12/dist-packages/transformers/models/grounding_dino/processing_grounding_dino.py:91: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)
Processing frames:  99%|█████████▉| 1329/1343 [20:03<00:12,  1.10frame/s, Blurs: 3 | Vehicles: 6 | ETA: 0:00:13] 


✅ Video processing complete!
  - Processed frames: 1329
  - Total blurs applied: 6636
  - Average blurs per frame: 4.99
  - Total vehicles detected: 21587
  - Processing time: 0:20:03
  - Output video: /kaggle/working/transformer_censored_video/censored_video_1080p.mp4
  - Output resolution: 1920x1080

🎬 Video saved at: /kaggle/working/transformer_censored_video/censored_video_1080p.mp4

📥 Download the processed video:


/kaggle/working/transformer_censored_video/censored_video_1080p.mp4


🎥 Preview of processed video (first 30 seconds):
